# 可选流程：Embedding × Ranker 组合评估

## Goal
完成单 logit CrossEncoder 的相关性微调，并把召回和重排的贡献分开评估。
默认用随机 tiny BERT 与同一套合成 corpus/query/qrels，执行真实 embedding、ranker 训练。
**指标只验证流程，不能证明业务效果。**

## Setup
仓库根目录：`python -m pip install -e '.[embedding,ranker,notebooks]'`。
Colab：`%pip install -e '/content/advai-notebooks[embedding,ranker,notebooks]'`，按下方设置 ROOT。

In [ ]:
from pathlib import Path
import sys
import importlib.metadata

# Colab: set ROOT = Path("/content/advai-notebooks") after placing the repo there.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Set ROOT to the repository directory before continuing")
sys.path.insert(0, str(ROOT / "src"))
for name in ["torch", "transformers", "sentence-transformers"]:
    print(name, importlib.metadata.version(name))
print("Repository:", ROOT)
from huggingface_hub.utils import disable_progress_bars
from transformers.utils.logging import disable_progress_bar
disable_progress_bars()
disable_progress_bar()


## Step 1 — 配置与统一数据
两种模型各自读取自己的训练配置与预生成数据，独立训练。完成后由 workflows 层加载导出模型做组合检索评估；组合步骤不会成为任何一侧训练的依赖。

In [ ]:
import tempfile
from advai_notebooks.embedding.config import load_config as load_embedding_config
from advai_notebooks.ranker.config import load_config as load_ranker_config
from advai_notebooks.common.retrieval_data import load_data
from advai_notebooks.embedding.pipeline import run as run_embedding
from advai_notebooks.ranker.pipeline import run as run_ranker

parent = ROOT / "outputs/retrieval"
parent.mkdir(parents=True, exist_ok=True)
EXPERIMENT = Path(tempfile.mkdtemp(prefix="notebook-", dir=parent))
embedding_cfg = load_embedding_config(ROOT / "configs/embedding/smoke.yaml", root=ROOT)
embedding_cfg.output_dir = str(EXPERIMENT / "embedding")
ranker_cfg = load_ranker_config(ROOT / "configs/ranker/smoke.yaml", root=ROOT)
ranker_cfg.output_dir = str(EXPERIMENT / "ranker")
print(load_data(ranker_cfg).summary())
print("Experiment:", EXPERIMENT)

## Step 2 — 独立训练 embedding
候选只由模型召回产生，不插入 ground truth。组合评估会为每种 embedding 独立召回候选，并让两个 ranker 复用该候选；ranker 自身训练读取预先保存的候选文件。

In [ ]:
embedding_report = run_embedding(embedding_cfg)
print("Independent embedding export:", embedding_report["export"])

## Step 3 — 训练 ranker、验证集选择与测试
训练正例来自 relevance>0 的 qrels，负例来自独立准备的训练样本，使用 BCEWithLogitsLoss。不同正例等级不会进入二分类目标，但评估 nDCG 保留等级信息。

epoch 0 也参与 best 选择。模型得分是排序 logit，未进行概率校准。

In [ ]:
ranker_report = run_ranker(ranker_cfg)
print("Selected epoch:", ranker_report["training"]["best_epoch"])
metric = ranker_cfg.evaluation.selection_metric
print("Baseline", metric, ranker_report["baseline_test"]["metrics"][metric])
print("Selected", metric, ranker_report["finetuned_test"]["metrics"][metric])

## Checks — 候选一致性与六组对照
两种 embedding 分别独立召回；每一种 embedding 的候选同时交给基线和训练后 ranker。另报两种 embedding 单独召回的指标，总共六行。漏召回正例保持缺失，Recall 分母仍含这些正例。

In [ ]:
import json
from IPython.display import display, HTML
from advai_notebooks.workflows.retrieval import compare
from advai_notebooks.workflows.config import load_config as load_workflow_config
workflow_cfg = load_workflow_config(ROOT / "configs/workflows/smoke.yaml")

ranker_dir = Path(ranker_cfg.output_dir)
candidates = json.loads((ranker_dir / "candidates.json").read_text())["splits"]["test"]
rankings = json.loads((ranker_dir / "test_rankings.json").read_text())
for qid, rows in candidates.items():
    ids = {r["doc_id"] for r in rows}
    assert ids == {r["doc_id"] for r in rankings["baseline"][qid]}
    assert ids == {r["doc_id"] for r in rankings["finetuned"][qid]}
comparison = compare(workflow_cfg, embedding_cfg.output_dir, ranker_cfg.output_dir,
                     EXPERIMENT / "comparison.json")
K = 3 if 3 in ranker_cfg.evaluation.cutoffs else (10 if 10 in ranker_cfg.evaluation.cutoffs else ranker_cfg.evaluation.cutoffs[0])
columns = ["embedding", "ranker", f"recall@{K}", f"mrr@{K}", f"ndcg@{K}"]
html = "<table><tr>" + "".join(f"<th>{k}</th>" for k in columns) + "</tr>"
for row in comparison["rows"]:
    html += "<tr>" + "".join(f"<td>{row[k]:.4f}</td>" if isinstance(row[k], float)
                            else f"<td>{row[k]}</td>" for k in columns) + "</tr>"
display(HTML(html + "</table>"))
print("Fixed candidates verified; six-way comparison saved.")

In [ ]:
import matplotlib.pyplot as plt
rows = comparison["rows"]
labels = [f"{r['embedding']} emb / {r['ranker']} ranker" for r in rows]
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(labels, [r[f"ndcg@{K}"] for r in rows], color=["#245f9b", "#75a4cf", "#37a888"] * 2)
ax.set(xlabel=f"Test nDCG@{K}", xlim=(0, 1.05),
       title="Synthetic smoke comparison; selected exports may equal baseline")
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(EXPERIMENT / "comparison.png", dpi=130)
plt.show()

## Step 4 — 重载模型并验证原生 SDK
wrapper 对 query/document 前缀各应用一次。SDK 直接调用时，先按导出 metadata 处理前缀，再显式指定 Identity 激活取得原始 logit。

In [ ]:
import numpy as np
import torch
from sentence_transformers import CrossEncoder
from advai_notebooks.ranker.model import RankerModel

model = RankerModel.load(ranker_report["export"], device="cpu")
sdk = CrossEncoder(ranker_report["export"], device="cpu")
pairs = [("reset password", "account login password reset"),
         ("reset password", "payment invoice card refund")]
model.check_lengths(pairs)
scores = model.score(pairs)
expected = sdk.predict(model.prepare(pairs), activation_fn=torch.nn.Identity(), show_progress_bar=False).reshape(-1)
np.testing.assert_allclose(scores, expected, atol=1e-6)
print("Reload and SDK agreement passed.")
for pair, score in sorted(zip(pairs, scores), key=lambda p: -p[1]):
    print(f"{score:.5f}", pair)
print("Report:", ranker_dir / "report.json")

## Next steps
- 使用业务数据与合适的独立 query/source group 划分；默认只采用明确的负例，未标注不等于不相关。
- 切换 `configs/ranker/bge.yaml`，设定一维相关性分类器与准备好的样本、候选文件路径。
- 比较同一候选集的 ranker 改善，也观察召回 Recall ceiling；用真实查询测延迟、吞吐和 GPU 显存。
- 根据业务需求扩展 pairwise/listwise 目标、模型压缩或概率校准。

v1 支持单设备全参数训练和推理 checkpoint 导出；尚未实现优化器状态恢复。真实大模型、CUDA/MPS 的效果和资源占用需在目标硬件验证。